In [90]:
import nest_asyncio
import time
import requests
import pandas as pd
from datetime import datetime
from io import BytesIO
from playwright.async_api import async_playwright
from openpyxl import load_workbook
from openpyxl.drawing.image import Image as ExcelImage
from openpyxl.styles import Alignment, Font
from openpyxl.utils import get_column_letter
from IPython.display import display

nest_asyncio.apply()

In [91]:
CLIENT_SECRET="amzn1.oa2-cs.v1.a8f04cdd3595029ad594e3535c0d0a59beb8c4500cd6bac96458f7b33ec5d0b6"
CLIENT_ID= "amzn1.application-oa2-client.5bd02b451f8d477693c5b99f64686ca7"
LWA_URL = "https://api.amazon.com/auth/o2/token"

In [92]:
TARGET_ACCOUNT = "LUFFCABO"       # E.g., 'BundWest' or 'LUFFCABO'
TARGET_MARKETPLACE = "United States"  # E.g., 'Australia', 'United States'
INPUT_EXCEL_FILE = "ASIN.xlsx"

In [93]:
CONFIG_JSON = {
  "sp_api_configurations": [
    {
      "account_name": "BundWest",
      "region": "Far East (FE)",
      "endpoint": "https://sellingpartnerapi-fe.amazon.com",
      "aws_region": "us-west-2",
      "refresh_token": "Atzr|IwEBIEMWwCh_GivH8L_felQ857L8hRUiE0dvK79QyAIxhmYHRdNf0mKD2Tn2T6M5RM_eZdDEWkBXEgRDxR4sKLOn1GmsDMNnbTXRhi7vp007_9H0HDF24c_RuN6iv3yxo7Mp9Jj2liqCGxfjm1Jy5ftO-3MqLriNzzAW-YJ_ud7KyvXjPmXSSf4k4SR1YakLG049iuvrwpuDIFVF0jnRgfPC1TgvpQj1U65XDk6JJiTFVCOBEcFqZ0I0W_j3ciyNscS72dMG7C9xyJIXMhIKPUxzkPqohSaBLtiEHsv2D9qkhCxRV8oGsbO5y5bm6b15iiJL0dTg-tEMcGi_-8fDEIVfGjtC",
      "marketplaces": {
        "Australia": "A39IBJ37TRP1C6"
      }
    },
    {
      "account_name": "BundWest",
      "region": "Europe (EU)",
      "endpoint": "https://sellingpartnerapi-eu.amazon.com",
      "aws_region": "eu-west-1",
      "refresh_token": "Atzr|IwEBIFq40drbKdAbotvgQsqIFoXtv9fA6qz_rUnfuiMwfL8ZCb8PsGqG1ddt4NjiAOyVBXcXpDACZpX7qMmvW1WsDiO2Bhu35Z5wafXhhFlIqlnFgPGqGaMDZik2MNS0Ddy5doUZ2jPggyLJ5dN1eMYL3s0KUyWLP4--TO4RciE9B-RPRgeZ8AyJYyPy5FezybEtQu2NolWc26didKweovoa3k14F0a_SJA2tSCefFW-fDjx-W8SlCcoutR68AJbuQNaoiDkgSRLaru99w6zQqE2oUSa9dWKPTZFAD8G7Xh7mmSkvQWG4jGim0-PPwsikZvgIl19RU1S6jaTpY1T0bTUriQt",
      "marketplaces": {
        "UAE": "A2VIGQ35RCS4UG"
      }
    },
    {
      "account_name": "BundWest",
      "region": "Europe (EU)",
      "endpoint": "https://sellingpartnerapi-eu.amazon.com",
      "aws_region": "eu-west-1",
      "refresh_token": "Atzr|IwEBIDJhNGj8yHTEkIM7kjrfH7U7Py-U9cCZPdiMl4H_mWgT9BFcN93tquTj3b9-iNkIIN3YFkDXyVUUIGtOJl3jZK99bYhcPz6egrLHKX6fmcujhZ8z2b5K3Q7J8ArT_0xmJTHN_VsWy2B3vTDjhYQI8aNKRG4GUSzt4gKkCHcsazfRuJmbZkRseF4eIiVFrfZ-bnS-w5sFMIMRgymlp9Od6pE9tjXLMdfIGig3lQzWSmDKYRMNDFlL2MDl47puf0L2oyBd6yw1kehVHQMSQENVcjV8tXiKgDJwO7rshYP6z7WQWUPcJFSHtip8Vb_SNji70i_FThUmv7XoOPxnwxrjYPY4",
      "marketplaces": {
        "Saudi Arabia": "A17E79C6D8DWNP"
      }
    },
    {
      "account_name": "BundWest",
      "region": "Europe (EU - Pan-EU Group)",
      "endpoint": "https://sellingpartnerapi-eu.amazon.com",
      "aws_region": "eu-west-1",
      "refresh_token": "AAtzr|IwEBIEPGn8_nHhCWKEvLJpkcyuwHMR2M1LXqVv65OxnEMpOsOo_zZqaUA1sH4s3iVc9Y9PnUBD1dSdo6UxmtR8p92JRweDokqC7StlR_WmJxMAPJ-tI3VS02wI7J1FnTHWRenHNbnZ2IfxZKAvaU9YxKfwuixq-nWRltqnol9RKnEUYT3cM_8C7mEmuWbdeb475xbCXiNy7r9o8cJ6WU9qgf5i0suMoqCz07KFuyQfMjrbwmGrztK0S8s5TJPwIERwGhVF0AbtoaXb0VDFDsi3-o-3I-A2Il-mijW7pjsZmz6uD8N55kM7KqIT5EM-KK1ucPgNAMNeuvlza0S8M6Ea1CsLXX",
      "marketplaces": {
        "United Kingdom": "A1F83G8C2ARO7P",
        "Germany": "A1PA6795UKMFR9",
        "France": "A13V1IB3VIYZZH",
        "Italy": "APJ6JRA9NG5V4",
        "Spain": "A1RKKUPIHCS9HS",
        "Netherlands": "A1805IZSGTT6HS",
        "Sweden": "A2NODRKZP88ZB9",
        "Poland": "A1C3SOZRARQ6R3",
        "Belgium": "AMEN7PMS3EDWL",
        "Ireland": "A1VC38T7YXB528"
      }
    },
    {
      "account_name": "LUFFCABO",
      "region": "North America (NA)",
      "endpoint": "https://sellingpartnerapi-na.amazon.com",
      "aws_region": "us-east-1",
      "refresh_token": "Atzr|IwEBIM5gyn0eF29KHBDaZThzkJ9JA0jnfiRuQBgq6OmC2bgGWuzYmauKALiZu7sz2t6GaFxrKvf-LHsk6LBN7FBOCI9u1pR2TEoLiXSjOgnjVdIk6Kihamle8sIX1mKUxCPoDVhzCrW5k1ZRwmrRkYUoPFU2DkPU50Hui7IRhv-SO-7SSOy-GVhNEVSWddfyhB53DTQKgByvME3zovDKmT1oboYFc96MDQC6Tps8vXPKg4WzzYNSnUluLW_X35ulx67k1SmF2KT-Fikgfuo0iI_jrwwwDFzwL0rUqhQ_Fk0zbdPilgA83L65H9Zp2rgfWEMZDz1o3TKiLaXBRBPYYs4aO_GE",
      "marketplaces": {
        "United States": "ATVPDKIKX0DER",
        "Canada": "A2EUQ1WTGCTBG2",
        "Mexico": "A1AM78C64UM0Y8",
        "Brazil": "A2Q3Y263D00KWC"
      }
    }
  ]
}

In [94]:
def get_access_token(refresh_token):
    payload = {
        "grant_type": "refresh_token",
        "refresh_token": refresh_token,
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET
    }
    headers = {"Content-Type": "application/x-www-form-urlencoded"}
    response = requests.post(LWA_URL, data=payload, headers=headers)
    response.raise_for_status()
    return response.json().get("access_token")

In [95]:
def get_target_config(account_name, marketplace_name):
    for config in CONFIG_JSON["sp_api_configurations"]:
        if config["account_name"] == account_name and marketplace_name in config["marketplaces"]:
            return config, config["marketplaces"][marketplace_name]
    raise ValueError("Invalid account or marketplace selection in config.")

In [96]:
async def scrape_amazon_frontend(asin, browser):
    """Captures visual data and checks for WAF blocks, redirects, and stock status."""
    url = f"https://www.amazon.com/dp/{asin}"
    
    context = await browser.new_context(
        user_agent="Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        viewport={"width": 1280, "height": 1080}
    )
    await context.add_init_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    
    page = await context.new_page()
    try:
        # 1. Faster Loading: Wait for DOM (not full network idle) to prevent timeouts
        await page.goto(url, timeout=60000, wait_until="domcontentloaded")
        await page.wait_for_timeout(3000)

        # 2. Hard CAPTCHA Check
        page_title = await page.title()
        if "Bot Check" in page_title or "Robot Check" in page_title:
            print(f"  -> HARD CAPTCHA BLOCKED for {asin}.")
            return True, "CAPTCHA Block", "CAPTCHA Block", "CAPTCHA Block", None

        # 3. WAF Auto-Clicker (Wrapped safely to prevent crashes)
        try:
            waf_button = await page.query_selector("text='Continue shopping'")
            if waf_button:
                await waf_button.click(timeout=5000)
                await page.wait_for_timeout(4000)
        except Exception:
            pass

        # 4. Strict Redirect Check
        if asin.upper() not in page.url.upper():
            print(f"  -> SKIPPING: {asin} redirected.")
            return False, "N/A", "N/A", "N/A", None

        # 5. Broad Out-of-Stock Check
        is_out_of_stock = False
        if await page.query_selector("#outOfStock"):
            is_out_of_stock = True
        
        availability_elem = await page.query_selector("#availability")
        if availability_elem:
            text = await availability_elem.inner_text()
            if "unavailable" in text.lower() or "we don't know when" in text.lower():
                is_out_of_stock = True

        if is_out_of_stock:
            print(f"  -> SKIPPING: {asin} is currently unavailable.")
            # Returning False triggers the blank "Unavailable" row in Excel
            return False, "N/A", "N/A", "N/A", None

        # 6. Null-Safe Metrics Extraction
        rating, reviews, coupon = "N/A", "N/A", "No Coupon"

        rating_elem = await page.query_selector("#acrPopover")
        if rating_elem:
            title_attr = await rating_elem.get_attribute("title")
            if title_attr:
                rating = title_attr.replace("out of 5 stars", "").strip()

        review_elem = await page.query_selector("#acrCustomerReviewText")
        if review_elem:
            rev_text = await review_elem.inner_text()
            if rev_text:
                reviews = rev_text.split(" ")[0].strip()

        coupon_elem = await page.query_selector('.promoPriceBlockMessage, label[id^="couponBadge"]')
        if coupon_elem:
            coup_text = await coupon_elem.inner_text()
            if coup_text:
                coupon = coup_text.strip().replace("\n", " ")

        # 7. Capture Screenshot
        screenshot_bytes = await page.screenshot()

    except Exception as e:
        print(f"  -> PIPELINE ERROR for {asin}: {e}")
        # Will visibly print "Scrape Error" in Excel so you know it crashed
        return True, "Scrape Error", "Scrape Error", "Scrape Error", None  
    finally:
        await page.close()
        await context.close()
        
    return True, rating, reviews, coupon, screenshot_bytes

In [97]:
# ==========================================
# 2. Async SP-API Data Fetcher (Backend)
# ==========================================
async def process_asin(asin, marketplace_id, region_name, endpoint, access_token, browser):
    # Ensures 'Region' shows the country name and non-available items remain blank
    base_row = {
        "ID": asin, "Region": region_name, "Brand": "", "Product Name": "",
        "Product URL": "", "Coupon Info": "", "Prime": "", "Rating": "",
        "Review Count": "", "Status": "Unavailable", "Product Screenshot": "", "Ranking": "",
        "Screenshot_Bytes": None
    }

    headers = {"x-amz-access-token": access_token}
    cat_url = f"{endpoint}/catalog/2022-04-01/items/{asin}"
    cat_res = requests.get(cat_url, headers=headers, params={"marketplaceIds": marketplace_id, "includedData": "summaries,salesRanks"})

    if cat_res.status_code != 200:
        return base_row 

    print(f"  -> Launching browser for {asin}...")
    is_match, r_rating, r_reviews, r_coupon, screen_bytes = await scrape_amazon_frontend(asin, browser)
    
    # Abort if the item redirected or is out of stock
    if not is_match:
        return base_row

    row = base_row.copy()
    row["Status"] = "Active"
    row["Product URL"] = f"https://www.amazon.com/dp/{asin}"
    row["Rating"] = r_rating
    row["Review Count"] = r_reviews
    row["Coupon Info"] = r_coupon
    row["Screenshot_Bytes"] = screen_bytes  

    item_data = cat_res.json()
    if item_data.get("summaries"):
        row["Brand"] = item_data["summaries"][0].get("brand", "")
        row["Product Name"] = item_data["summaries"][0].get("itemName", "")

    ranks_data = item_data.get("salesRanks", [])
    if ranks_data and "displayGroupRanks" in ranks_data[0]:
        row["Ranking"] = ", ".join([f"#{r.get('rank')} in {r.get('title')}" for r in ranks_data[0]["displayGroupRanks"]])

    price_url = f"{endpoint}/products/pricing/v0/items/{asin}/offers"
    price_res = requests.get(price_url, headers=headers, params={"MarketplaceId": marketplace_id, "ItemCondition": "New"})
    
    if price_res.status_code == 200:
        offers = price_res.json().get("payload", {}).get("Offers", [])
        row["Prime"] = "Yes" if any(o.get("IsFulfilledByAmazon") for o in offers) else "No"
        
    return row


In [98]:
async def main_notebook_execution():
    print(f"Starting job for {TARGET_ACCOUNT} - {TARGET_MARKETPLACE}...")
    
    try:
        config, marketplace_id = get_target_config(TARGET_ACCOUNT, TARGET_MARKETPLACE)
        access_token = get_access_token(config["refresh_token"])
        
        input_df = pd.read_excel(INPUT_EXCEL_FILE)
        asin_col = next((col for col in input_df.columns if str(col).strip().upper() in ["ASIN", "ID", "PRODUCT ID"]), input_df.columns[0])
        asins = [str(val).strip().upper() for val in input_df[asin_col].dropna().tolist() if str(val).strip()]
        
        print(f"Found {len(asins)} ASINs. Launching browser instance...\n")
        processed_rows = []
        
        # Launch Playwright browser exactly once to process all ASINs rapidly
        async with async_playwright() as p:
            browser = await p.chromium.launch(headless=False, args=["--disable-blink-features=AutomationControlled"])
            
            for idx, asin in enumerate(asins, 1):
                print(f"[{idx}/{len(asins)}] Processing: {asin}")
                # Pass TARGET_MARKETPLACE so the real country name populates the 'Region' column
                row_data = await process_asin(asin, marketplace_id, TARGET_MARKETPLACE, config["endpoint"], access_token, browser)
                processed_rows.append(row_data)
                time.sleep(0.3)
                
            await browser.close()
            
        print("\nFormatting Excel file...")

        # Drop raw bytes, reorder columns
        columns_order = ["ID", "Region", "Brand", "Product Name", "Product URL", "Coupon Info", "Prime", "Rating", "Review Count", "Status", "Product Screenshot", "Ranking"]
        export_data = [{k: v for k, v in r.items() if k != "Screenshot_Bytes"} for r in processed_rows]
        output_df = pd.DataFrame(export_data)[columns_order]
        
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_filename = f"Processed_{TARGET_MARKETPLACE}_ASINs_{timestamp}.xlsx"
        output_df.to_excel(output_filename, index=False)

        # Apply openpyxl layout restrictions
        wb = load_workbook(output_filename)
        ws = wb.active

        center_aligned_text = Alignment(horizontal="center", vertical="center", wrap_text=True)
        for row in ws.iter_rows():
            for cell in row:
                cell.alignment = center_aligned_text

        for cell in ws[1]:
            cell.font = Font(bold=True)
        ws.row_dimensions[1].height = 30

        for col_idx in range(1, ws.max_column + 1):
            col_letter = get_column_letter(col_idx)
            ws.column_dimensions[col_letter].width = 25

        img_col_idx = next((col for col in range(1, ws.max_column + 1) if ws.cell(row=1, column=col).value == "Product Screenshot"), None)

        for row_idx in range(2, ws.max_row + 1):
            ws.row_dimensions[row_idx].height = 105

            if img_col_idx:
                cell = ws.cell(row=row_idx, column=img_col_idx)
                screen_bytes = processed_rows[row_idx - 2].get("Screenshot_Bytes")

                if screen_bytes:
                    try:
                        img_data = BytesIO(screen_bytes)
                        excel_img = ExcelImage(img_data)
                        excel_img.width = 150
                        excel_img.height = 100
                        excel_img.anchor = f"{get_column_letter(img_col_idx)}{row_idx}"
                        ws.add_image(excel_img)
                    except Exception:
                        cell.value = "Render Error"
                else:
                    cell.value = "Unavailable"

        wb.save(output_filename)
        print(f"Success! Final formatted file saved locally as: {output_filename}")
        display(output_df.head())

    except Exception as e:
        print(f"\nERROR ALONG PIPELINE: {e}")

# This await command executes the entire script seamlessly inside Jupyter
await main_notebook_execution()

Starting job for LUFFCABO - United States...
Found 21 ASINs. Launching browser instance...

[1/21] Processing: B0C2Z188J6
[2/21] Processing: B07LF8QXVQ
[3/21] Processing: B0CLGCN5ZS
  -> Launching browser for B0CLGCN5ZS...
[4/21] Processing: B092G3K1SV
  -> Launching browser for B092G3K1SV...
  -> SKIPPING: B092G3K1SV redirected.
[5/21] Processing: B07X5HG2HC
  -> Launching browser for B07X5HG2HC...
  -> SKIPPING: B07X5HG2HC is currently unavailable.
[6/21] Processing: B0FDWMNFCT
[7/21] Processing: B07X3BQ7P7
[8/21] Processing: B01HF5SYUK
  -> Launching browser for B01HF5SYUK...
  -> SKIPPING: B01HF5SYUK is currently unavailable.
[9/21] Processing: B0D97H4W9Q
  -> Launching browser for B0D97H4W9Q...
[10/21] Processing: B0H1W6RKTJ
[11/21] Processing: B0DDJQPND3
[12/21] Processing: B09F3PH897
[13/21] Processing: B08HN3DQV6
[14/21] Processing: B0CVGYRKSP
[15/21] Processing: B094774HD3
[16/21] Processing: B09R6B4STG
[17/21] Processing: B0793D8QSZ
  -> Launching browser for B0793D8QSZ...
  

,ID,Region,Brand,Product Name,Product URL,Coupon Info,Prime,Rating,Review Count,Status,Product Screenshot,Ranking
0,B0C2Z188J6,United States,,,,,,,,Unavailable,,
1,B07LF8QXVQ,United States,,,,,,,,Unavailable,,
2,B0CLGCN5ZS,United States,Momcozy,"Momcozy PureHug Baby Carrier - Ergonomic, Cozy...",https://www.amazon.com/dp/B0CLGCN5ZS,No Coupon,Yes,4.5,"(5,694)",Active,,#327 in Baby
3,B092G3K1SV,United States,,,,,,,,Unavailable,,
4,B07X5HG2HC,United States,,,,,,,,Unavailable,,
